# Lab 04-03 — Hybrid retrieval: fusing dense (vector) + sparse (BM25) rankings

**Track 04 · Retrieval** — dense retrieval (embedding similarity) is strong on *semantic* matches: it finds the passage that means the same thing as the question even when the wording shares no words. Sparse retrieval (BM25) is strong on *exact term* matches: it nails the passage that literally contains the rare keyword the question names. Each arm alone is wrong in a different way — so we fuse them.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the BM25 arm, and both fusion paths appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

This lab builds both arms over the same deterministic subset of `Data/corpus/rag-mini-wikipedia`:

* **DENSE arm** — `FAISS` store + `as_retriever` (BGE embeddings, cosine-style top-k).
* **SPARSE arm** — `BM25Retriever` (rank_bm25 backend): token-overlap top-k.

Then it fuses the two rankings with Reciprocal Rank Fusion (RRF): every document earns `weight / (c + rank)` for each ranked list it appears in, and the summed scores decide the final order. A document found by BOTH arms collects two contributions and is deduplicated — the classic hybrid win.

Two fusion implementations are demonstrated side by side:

* `EnsembleRetriever` (langchain-classic, the plan-preferred path) — the dense arm is already a Runnable, weights `[0.5, 0.5]`, `c=60`.
* an **inline RRF fusion class** — the same RRF math (`1/(60+rank)`), dedup by `page_content`, consuming the sparse arm through a tiny `BM25Adapter`; this is the exact code of the repo's `HybridRetriever` (`src/retrieval/hybrid.py`) written inline instead of imported.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-classic`, `langchain-community`, `langchain-huggingface`, `sentence-transformers`, `faiss-cpu`, `rank-bm25`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS index (langchain_community)
#   langchain-classic     -> EnsembleRetriever (the plan-preferred fusion)
#   rank-bm25             -> the BM25 sparse arm (langchain_community)
#   pandas                -> reads the passages/test.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community langchain-classic faiss-cpu rank-bm25 pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_classic.retrievers.ensemble import EnsembleRetriever  # noqa: E402
from langchain_community.retrievers.bm25 import BM25Retriever  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes the deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610, 1604]` are real questions whose answers live inside the subset, plus one purely semantic question (no rare keyword shared with its target passage — passages 44/6 say "88% of the population are of European descent", the question never says "European descent", so only the dense arm can bridge it). `TOP_K = 5` is the budget of every arm and fusion; `RRF_C = 60` is the RRF constant and `WEIGHTS = [0.5, 0.5]` the dense:sparse fusion weights.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
# A purely semantic question (no rare keyword shared with its target passage):
# passages 44/6 say "88% of the population are of European descent" — the
# question never says "European descent", so only the dense arm can bridge it.
SEMANTIC_QUESTION = "What share of the nation's people trace their roots to Europe?"
TOP_K = 5
RRF_C = 60  # RRF constant: how strongly lower ranks are discounted (k=60 is standard)
WEIGHTS = [0.5, 0.5]  # dense : sparse fusion weights
PREVIEW = 62  # max characters of passage text shown next to each hit
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` pulls the first `n` passages (text + ids) from `passages.parquet`; `load_questions` pulls specific rows by id from `test.parquet`; `preview` flattens a passage onto one line. Three small classes/helpers round out the section: `BM25Adapter` exposes the classic BM25 retriever as a plain `.retrieve(question)` contract (it is a Runnable and speaks `.invoke()`); `_RRFHybridRetriever` is the inline RRF fusion with the exact math of the repo's `HybridRetriever` (dedup by `page_content`, `1/(60+rank)` contributions); and `overlap_analysis` computes which texts the two arms agree on.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions + the two fusion helpers
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


class BM25Adapter:
    """Expose ``BM25Retriever`` as a plain ``.retrieve(question)`` contract.

    The inline RRF fusion class calls ``sparse_retriever.retrieve(question)``;
    the classic BM25 retriever is a Runnable and speaks ``.invoke()`` instead.
    This tiny adapter bridges the two so the fusion consumes the same sparse
    arm the EnsembleRetriever path uses.
    """

    def __init__(self, bm25: BM25Retriever):
        self._bm25 = bm25

    def retrieve(self, question: str) -> list[Document]:
        return self._bm25.invoke(question)


class _RRFHybridRetriever:
    """Inline RRF fusion — the same math as the repo's ``HybridRetriever``.

    Every document earns 1 / (60 + rank) for each ranked list it appears in,
    and the summed scores decide the final order. Documents found by both
    arms are deduplicated (key = page_content) and get the two-contribution
    boost — the classic hybrid win. Written inline instead of imported from
    ``src/retrieval/hybrid.py``.
    """

    # RRF constant: how strongly lower ranks are discounted (k=60 is standard).
    RRF_K: int = 60

    def __init__(self, dense_retriever, sparse_retriever, top_k: int = 5):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.top_k = top_k

    @staticmethod
    def _doc_key(doc: Document) -> str:
        """RRF dedup key: the page content, or a content hash when empty."""
        return doc.page_content or str(hash(repr(doc)))

    def retrieve(self, question: str) -> list[Document]:
        """Run both arms, fuse with RRF, return the top-k documents."""
        dense_docs = self.dense_retriever.retrieve(question)
        sparse_docs = self.sparse_retriever.retrieve(question)

        scores: dict[str, float] = {}
        by_key: dict[str, Document] = {}

        for ranked in (dense_docs, sparse_docs):
            for rank, doc in enumerate(ranked, start=1):
                key = self._doc_key(doc)
                scores[key] = scores.get(key, 0.0) + 1.0 / (self.RRF_K + rank)
                by_key[key] = doc

        fused = sorted(by_key, key=lambda key: scores[key], reverse=True)
        return [by_key[key] for key in fused[: self.top_k]]


def overlap_analysis(
    dense_docs: list[Document], sparse_docs: list[Document]
) -> tuple[dict[str, int], dict[str, int], list[str]]:
    """Return (dense_rank_by_text, sparse_rank_by_text, texts_found_by_both)."""
    dense_rank = {d.page_content: r for r, d in enumerate(dense_docs, 1)}
    sparse_rank = {d.page_content: r for r, d in enumerate(sparse_docs, 1)}
    both = sorted(set(dense_rank) & set(sparse_rank))
    return dense_rank, sparse_rank, both


## 3. Experiment — embed, index, build both arms, fuse

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model; we embed the 100 passages once and hand the FAISS store its vectors through a tiny precomputed passthrough, so the embed step and the index step stay separately timed. The dense arm is the store's own `as_retriever` (a Runnable — it plugs straight into `EnsembleRetriever`); the sparse arm is `BM25Retriever.from_documents`. Fusion path 1 is the plan-preferred `EnsembleRetriever` with weights `[0.5, 0.5]`, `c=60`, `id_key=None` (dedup by page_content); fusion path 2 is the inline `_RRFHybridRetriever` over the same two arms via adapters. Every question — the three real ones plus the semantic one — runs through both arms and both fusions. `run_experiment` returns a dict of artifacts instead of printing, so the demo and the gate read the same run.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed, index, build both arms, fuse; returns every artifact
#    the demo and the verification gate need (no re-computation between paths)
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)
    questions.append((len(questions), SEMANTIC_QUESTION))  # 4th, purely semantic

    # --- Embed the whole subset once (batched) + each question once ---------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    question_texts = [qtext for _, qtext in questions]
    query_vecs = [embedder.embed_query(q) for q in question_texts]

# --- Build the FAISS index (in-memory) ----------------------------------
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    # from_embeddings builds the index from the precomputed vectors (so the
    # embed step and the index step stay separately timed) while keeping the
    # real embedder attached — store.embed_query() works for the retriever.
    t0 = time.perf_counter()
    store = FAISS.from_embeddings(
        list(zip(passage_texts, passage_vecs)),
        embedding=embedder,
        metadatas=[{"id": pid} for pid in passage_ids],
    )
    index_s = time.perf_counter() - t0

    # --- The two arms over the same corpus ----------------------------------
    dense = store.as_retriever(search_kwargs={"k": TOP_K})  # already a Runnable
    sparse = BM25Retriever.from_documents(chunks, k=TOP_K)

    # --- Fusion path 1: langchain-classic EnsembleRetriever (plan-preferred) -
    # id_key=None dedups by page_content (the same key the inline RRF uses).
    ensemble = EnsembleRetriever(
        retrievers=[dense, sparse],
        weights=WEIGHTS,
        c=RRF_C,
        k=TOP_K,
        id_key=None,
    )

    # --- Fusion path 2: inline RRF (1/(60+rank), same math as the repo) -----
    class _DenseAdapter:
        """Expose the VectorStoreRetriever as a plain .retrieve() contract."""

        def __init__(self, retriever):
            self._retriever = retriever

        def retrieve(self, question: str) -> list[Document]:
            return self._retriever.invoke(question)

    inline_rrf = _RRFHybridRetriever(
        _DenseAdapter(dense), BM25Adapter(sparse), top_k=TOP_K
    )

    # --- Run every question through both arms and both fusions --------------
    per_question = []
    for qid, qtext in questions:
        dense_docs = dense.invoke(qtext)
        sparse_docs = sparse.invoke(qtext)
        # EnsembleRetriever returns the full RRF-ranked list (union of both
        # arms); take the top TOP_K — the same contract the inline RRF uses.
        fused_ens = ensemble.invoke(qtext)[:TOP_K]
        fused_rrf = inline_rrf.retrieve(qtext)
        dense_rank, sparse_rank, both = overlap_analysis(dense_docs, sparse_docs)
        per_question.append(
            {
                "qid": qid,
                "qtext": qtext,
                "dense": dense_docs,
                "sparse": sparse_docs,
                "fused_ens": fused_ens,
                "fused_rrf": fused_rrf,
                "dense_rank": dense_rank,
                "sparse_rank": sparse_rank,
                "both": both,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "query_vecs": query_vecs,
        "embed_s": embed_s,
        "index_s": index_s,
        "per_question": per_question,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with all four questions and the two arms' description; per question, the dense top-5, the sparse top-5, the fused top-5 through `EnsembleRetriever` and through the inline RRF, plus the "found by BOTH arms" lines showing the RRF boost with the rank each arm gave and the rank fusion produced; then a takeaway framing RRF as the fusion that makes both-arm documents win.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 03 — Hybrid retrieval: dense + sparse fusion (RRF)")
    print(f"{BGE_MODEL_NAME} dense | BM25 sparse | RRF c={RRF_C} weights={WEIGHTS}")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions ({len(exp['questions']) - 1} from test.parquet + 1 semantic):")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] The two arms (same corpus, same top-{TOP_K}):")
    print(f"    embedded {exp['indexed']} passages in {exp['embed_s']:.2f}s (dim {exp['dim']})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s")
    print("    dense  = FAISS + as_retriever               (semantic match)")
    print("    sparse = BM25Retriever (rank_bm25)          (exact-term match)")

    for q in exp["per_question"]:
        print(f'\n[3] Q[{q["qid"]}] "{q["qtext"]}"')
        print(f'    dense top-{TOP_K}:')
        for rank, doc in enumerate(q["dense"], 1):
            pid = doc.metadata.get("id", "?")
            print(f"      {rank}. [p{pid}] {preview(doc.page_content)}")
        print(f'    sparse top-{TOP_K}:')
        for rank, doc in enumerate(q["sparse"], 1):
            pid = doc.metadata.get("id", "?")
            print(f"      {rank}. [p{pid}] {preview(doc.page_content)}")
        print(f'    fused top-{TOP_K} (EnsembleRetriever):')
        for rank, doc in enumerate(q["fused_ens"], 1):
            pid = doc.metadata.get("id", "?")
            print(f"      {rank}. [p{pid}] {preview(doc.page_content)}")
        print(f'    fused top-{TOP_K} (inline RRF — same math as repo HybridRetriever):')
        for rank, doc in enumerate(q["fused_rrf"], 1):
            pid = doc.metadata.get("id", "?")
            print(f"      {rank}. [p{pid}] {preview(doc.page_content)}")
        if q["both"]:
            fused_rank = {d.page_content: r for r, d in enumerate(q["fused_ens"], 1)}
            print(f'    found by BOTH arms ({len(q["both"])} docs — RRF boost):')
            for text in q["both"]:
                pid = next(
                    d.metadata.get("id", "?")
                    for d in q["dense"] + q["sparse"]
                    if d.page_content == text
                )
                print(f"      [p{pid}] dense#{q['dense_rank'][text]} sparse#{q['sparse_rank'][text]} "
                      f"-> fused#{fused_rank.get(text, '-')}")
        else:
            print("    found by BOTH arms: none (arms agree on nothing this query)")

    print("\n[4] Takeaway")
    print("    Dense alone misses exact-term needles; sparse alone misses")
    print("    paraphrases. RRF fuses the two rankings: a doc found by both")
    print("    arms collects two 1/(60+rank) contributions, is deduplicated,")
    print("    and outranks every doc found by only one arm — the hybrid win.")
    print("    EnsembleRetriever (langchain-classic) and the inline RRF class")
    print("    implement the same RRF math and agree here.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: embedding dimension 768 and exactly `N_PASSAGES` passages indexed; both arms return exactly `TOP_K` docs per question; both fusions return exactly `TOP_K` docs with no duplicates; RRF cannot invent documents — every fused doc must come from the union of the two arms' result sets (checked for both fusion paths); the keyword-heavy query's fused top-1 contains the rare term 'montevideo'; the purely semantic query's fused top-1 is the European-descent passage; and the RRF boost check — a document found by BOTH arms outranks every document found by only one arm in the fused output. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Dimension and count match the model / subset.
    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Both arms return exactly TOP_K documents for every question.
    checks.append(
        ("dense arm returns exactly TOP_K docs per question",
         all(len(q["dense"]) == TOP_K for q in exp["per_question"]))
    )
    checks.append(
        ("sparse arm returns exactly TOP_K docs per question",
         all(len(q["sparse"]) == TOP_K for q in exp["per_question"]))
    )

    # Both fusions return exactly TOP_K documents with no duplicates.
    checks.append(
        ("EnsembleRetriever fused returns TOP_K docs, no duplicates",
         all(len(q["fused_ens"]) == TOP_K and len({d.page_content for d in q["fused_ens"]}) == TOP_K
             for q in exp["per_question"]))
    )
    checks.append(
        ("inline RRF fused returns TOP_K docs, no duplicates",
         all(len(q["fused_rrf"]) == TOP_K and len({d.page_content for d in q["fused_rrf"]}) == TOP_K
             for q in exp["per_question"]))
    )

    # RRF cannot invent documents: every fused doc must come from the union of
    # the two arms' result sets (checked for both fusion paths).
    def fused_within_union(fused: list[Document], dense: list[Document], sparse: list[Document]) -> bool:
        union = {d.page_content for d in dense} | {d.page_content for d in sparse}
        return all(d.page_content in union for d in fused)

    checks.append(
        ("every fused doc appears in the union of dense+sparse results (EnsembleRetriever)",
         all(fused_within_union(q["fused_ens"], q["dense"], q["sparse"]) for q in exp["per_question"]))
    )
    checks.append(
        ("every fused doc appears in the union of dense+sparse results (inline RRF)",
         all(fused_within_union(q["fused_rrf"], q["dense"], q["sparse"]) for q in exp["per_question"]))
    )

    # Keyword-heavy query: Q1610 "Who founded Montevideo?" — the fused top-1
    # must be the passage that literally contains the rare term "Montevideo"
    # (passage id 2: "Montevideo was founded by the Spanish ...").
    q1610 = next(q for q in exp["per_question"] if q["qid"] == 1610)
    kw_top1 = q1610["fused_ens"][0].page_content.lower()
    checks.append(("keyword query fused top-1 contains the rare term 'montevideo'", "montevideo" in kw_top1))

    # Purely semantic query: the fused top-1 must be the passage about European
    # descent (passage id 6) even though the question never says those words.
    sem = exp["per_question"][-1]
    sem_top1 = sem["fused_ens"][0].page_content.lower()
    checks.append(("semantic query fused top-1 is the European-descent passage", "european descent" in sem_top1))

    # RRF boost: a document found by BOTH arms outranks every document found by
    # only one arm in the fused output (two contributions beat one, always).
    boost_ok = True
    any_overlap = False
    for q in exp["per_question"]:
        if not q["both"]:
            continue
        any_overlap = True
        fused_rank = {d.page_content: r for r, d in enumerate(q["fused_ens"], 1)}
        both_ranks = [fused_rank[t] for t in q["both"] if t in fused_rank]
        single_ranks = [r for t, r in fused_rank.items() if t not in q["both"]]
        if both_ranks and single_ranks and max(both_ranks) > min(single_ranks):
            boost_ok = False
    checks.append(("RRF boost: both-arm docs outrank single-arm docs in fused output", boost_ok and any_overlap))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A couple of minutes of embedding + index build + BM25 fits — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

For each of the four questions: dense top-5, sparse top-5, and the fused top-5 through both fusion implementations, with the found-by-BOTH-arm lines showing where RRF boosts the overlap.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
